In [5]:
!pip install -q "transformers==4.46.3" datasets evaluate wandb accelerate

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 1.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 35.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 38.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 96.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [2]:
import wandb
wandb.login()

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: adam234 (adam234-datum-analysis) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [3]:
import numpy as np, evaluate, wandb
from datasets import load_dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer, DataCollatorWithPadding)

MODEL = "distilbert-base-uncased"
NUM_LABELS = 2

def main():
    wandb.init(project="ai-trainee", name="day03-distilbert-imdb")

    ds = load_dataset("stanfordnlp/imdb")
    # Use smaller subsets so training finishes in an hour on a 3060
    ds["train"] = ds["train"].shuffle(seed=42).select(range(8000))
    ds["test"]  = ds["test"].shuffle(seed=42).select(range(2000))

    tok = AutoTokenizer.from_pretrained(MODEL)

    def tokenize(batch):
        return tok(batch["text"], truncation=True, max_length=256)

    ds = ds.map(tokenize, batched=True)
    collator = DataCollatorWithPadding(tokenizer=tok)

    model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=NUM_LABELS)

    metric_acc = evaluate.load("accuracy")
    metric_f1  = evaluate.load("f1")
    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        preds = np.argmax(logits, axis=-1)
        return {
            "accuracy": metric_acc.compute(predictions=preds, references=labels)["accuracy"],
            "f1":       metric_f1.compute(predictions=preds, references=labels, average="macro")["f1"],
        }

    args = TrainingArguments(
        output_dir="models/day03_distilbert_imdb",
        num_train_epochs=2,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        eval_strategy="epoch",
        save_strategy="epoch",
        learning_rate=2e-5,
        weight_decay=0.01,
        warmup_ratio=0.1,
        logging_steps=50,
        fp16=True,           # use RTX 3060 tensor cores
        report_to="wandb",
        load_best_model_at_end=True,
        metric_for_best_model="f1",
    )

    trainer = Trainer(
        model=model, args=args,
        train_dataset=ds["train"], eval_dataset=ds["test"],
        tokenizer=tok, data_collator=collator,
        compute_metrics=compute_metrics,
    )
    trainer.train()
    trainer.save_model("models/day03_distilbert_imdb/final")

if __name__ == "__main__":
    main()

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Map:   0%|          | 0/8000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipykernel_7095/2913747870.py:54: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.265800,0.262517,0.891000,0.890769
2,0.224500,0.293455,0.901000,0.900928


In [1]:
import transformers
print(transformers.__version__)

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

4.46.3


In [4]:
!zip -r model.zip models/day03_distilbert_imdb/final
from google.colab import files
files.download("model.zip")

  adding: models/day03_distilbert_imdb/final/ (stored 0%)
  adding: models/day03_distilbert_imdb/final/training_args.bin (deflated 53%)
  adding: models/day03_distilbert_imdb/final/special_tokens_map.json (deflated 42%)
  adding: models/day03_distilbert_imdb/final/vocab.txt (deflated 53%)
  adding: models/day03_distilbert_imdb/final/model.safetensors (deflated 8%)
  adding: models/day03_distilbert_imdb/final/config.json (deflated 46%)
  adding: models/day03_distilbert_imdb/final/tokenizer_config.json (deflated 76%)
  adding: models/day03_distilbert_imdb/final/tokenizer.json (deflated 71%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [5]:
files.download("model.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>